# Siraj Uddin Mohammad (24089738)
### R1 — Integrator & Git Lead: Repository Setup, Schema Validation & Data Integrity
**Group P (`Siraj Group P`) — Dataset C (Nov 2026 Data Science Challenge Hackathon)**

*Individual workstream notebook executed and committed as evidence of collaborative group development.*

# Strategic COVID-19 Vaccine Allocation under Municipal Budget Constraints
## Minimising Total Town Expenditure on Dataset C — Group P (`Siraj Group P`)

| Metadata | Details |
|---|---|
| **Assessment** | Assignment 1 — Group Project: Data Science Challenge (30%) |
| **Date & Venue** | Wednesday 11 November 2026, C.D.I.O. 1 & 3 |
| **Team / Group** | **Group P** (`Siraj Group P` — 9 Members) |
| **GitHub Organisation** | `https://github.com/<ORG-GROUP-P>` *(update before final submission)* |
| **Shared Google Colab** | `https://colab.research.google.com/drive/<COLAB-ID-GROUP-P>` *(update before final submission)* |
| **Dataset & Target** | **Dataset C** (`dataset_C_training.csv`, $N_{\text{train}}=4,756$; `dataset_C_testing.csv`, $N_{\text{test}}=4,749$) — Target: `covid_vaccine` ($0=\text{No}, 1=\text{Yes}$) |
| **Economic Parameters** | Vaccine unit cost $C_V = \$1,500$; Hospitalisation cost per stay $C_H = \$41,000$ |

### Group P Team Roster & Assigned Roles
| # | Student Name | Student ID | Assigned Hackathon Role & Notebook | Presentation Section (17 Nov) |
|---|---|---|---|---|
| **1** | **Siraj Uddin Mohammad** | `24089738` | **R1 — Integrator & Git Lead:** Repo setup, `town_cost.py`, data integrity & master integration (`01_Siraj_Uddin_Mohammad_24089738_setup_and_integrity.ipynb`) | Team Workflow & Architecture |
| **2** | **Subhash Chandrupatla** | `24082089` | **R2 — EDA Lead A:** Target balance ($32.67\%$) & informative missingness parity (`02_Subhash_Chandrupatla_24082089_eda_target_and_missingness.ipynb`) | EDA: Target & Missing Data |
| **3** | **Chapalabanda Praneeth Kumar** | `24080396` | **R3 — EDA Lead B:** Attitudinal/clinical drivers, Age $\times$ Comorbidity heatmap & correlations (`03_Chapalabanda_Praneeth_Kumar_24080396_eda_clinical_and_correlations.ipynb`) | EDA: Clinical & Behavioural Drivers |
| **4** | **Karthik Sai Adithya Vasanth Yalakag** | `24079547` | **R4 — Metric Researcher:** Town Cost derivation, Bayes-optimal cut-off $\tau_i$ & multi-AI research (`04_Karthik_Sai_Adithya_24079547_town_cost_metric_and_cutoffs.ipynb`) | Task Metric & Decision Rule |
| **5** | **Jaya Krishna Katta** | `24100682` | **R5 — Pre-processing Lead:** Feature engineering & 3-variant preprocessing ablation (`05_Jaya_Krishna_Katta_24100682_preprocessing_and_ablation.ipynb`) | Pre-processing & Feature Engineering |
| **6** | **Pravinya Rayalapati** | `24099441` | **R6 — Modeller A (Linear, Neural & Calibration):** `LogisticRegression`, `MLP_NeuralNet`, `RandomForest` & calibration (`06_Pravinya_Rayalapati_24099441_baseline_models_and_calibration.ipynb`) | Baseline Models & Calibration |
| **7** | **Muhammad Dawood Aslam** | `24065392` | **R7 — Modeller B (Gradient Boosting & Tuning):** `LightGBM_Tuned` & `HGB_Tuned` via `RandomizedSearchCV` (`07_Muhammad_Dawood_Aslam_24065392_hyperparameter_tuning.ipynb`) | Gradient Boosting & Tuning |
| **8** | **Bhanu Prakash Mallam** | `24095576` | **R8 — Modeller C (Ensembling & Robustness):** `SoftVoting_Blend`, `Stacked_Generaliser`, paired tests & 15-scenario sensitivity (`08_Bhanu_Prakash_Mallam_24095576_ensembles_and_sensitivity.ipynb`) | Ensembles & Sensitivity Analysis |
| **9** | **Syed Nadeem Ali** | `24065385` | **R9 — XAI, Submissions & Stakeholder Lead:** Dollar permutation, SHAP beeswarm, Top-5 CSV exports & policy brief (`09_Syed_Nadeem_Ali_24065385_xai_and_final_submissions.ipynb`) | Explainable AI & Stakeholder Summary |

---

### Executive Roadmap
1. **Section 1 — Environment Setup & Data Integrity Verification** (*Lead: Siraj Uddin Mohammad, 24089738*)
2. **Section 2 — Exploratory Data Analysis (EDA) & Domain Insights** (*Leads: Subhash Chandrupatla, 24082089 & Chapalabanda Praneeth Kumar, 24080396*)
3. **Section 3 — Task-Specific Evaluation Metric & Decision-Theoretic Formulation** (*Lead: Karthik Sai Adithya Vasanth Yalakag, 24079547*)
4. **Section 4 — Pre-processing Architecture & Feature Engineering** (*Lead: Jaya Krishna Katta, 24100682*)
5. **Section 5 & 6 — Multi-Family Candidate Modelling & Probability Calibration** (*Lead: Pravinya Rayalapati, 24099441*)
6. **Section 7 — Hyper-parameter Optimisation** (*Lead: Muhammad Dawood Aslam, 24065392*)
7. **Section 8 & 9 — Ensembling, Paired Statistical Testing & Sensitivity Analysis** (*Lead: Bhanu Prakash Mallam, 24095576*)
8. **Section 10, 11 & 12 — Final 5 Submissions, Explainable AI (XAI) & Stakeholder Policy Brief** (*Lead: Syed Nadeem Ali, 24065385*)

## 1. Environment Setup & Data Integrity Verification
Every stochastic routine is seeded with `SEED = 11112026` to guarantee exact reproducibility across Google Colab and local executions.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from lightgbm import LGBMClassifier

SEED = 11112026
GROUP_ID = "P"
TARGET_COL = "covid_vaccine"
ID_COL = "respondent_id"

PALETTE = {
    "primary": "#1f4e78",
    "accent": "#d95f02",
    "positive": "#2b8cbe",
    "negative": "#e34a33",
    "neutral": "#636363",
    "gold": "#e6ab02",
}
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams.update({
    "figure.dpi": 115,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

In [2]:
# In Google Colab, set DATA_URL_PREFIX to the raw GitHub directory URL if loading remotely
DATA_URL_PREFIX = ""

train_df = pd.read_csv(DATA_URL_PREFIX + "dataset_C_training.csv")
test_df = pd.read_csv(DATA_URL_PREFIX + "dataset_C_testing.csv")
example_sub = pd.read_csv(DATA_URL_PREFIX + "dataset_C_example_submission.csv")

y_train = train_df[TARGET_COL].to_numpy(dtype=int)
n_train, n_test = len(train_df), len(test_df)

print(f"Training set shape : {train_df.shape} (residents: {n_train:,})")
print(f"Testing set shape  : {test_df.shape} (residents: {n_test:,})")
print(f"Example submission : {example_sub.shape} | columns: {list(example_sub.columns)} | target dtype: {example_sub[TARGET_COL].dtype}")

# Integrity checks
assert ID_COL in train_df.columns and ID_COL in test_df.columns, "Missing respondent_id column"
assert set(train_df.columns) - {TARGET_COL} == set(test_df.columns), "Train/Test feature mismatch"
assert example_sub[ID_COL].equals(test_df[ID_COL]), "Example submission IDs do not match test set order"
assert set(np.unique(y_train)) == {0, 1}, "Target must be binary 0/1"
assert len( set(train_df[ID_COL]).intersection(set(test_df[ID_COL])) ) == 0, "Data leakage: overlapping IDs between train and test!"

display(train_df.head(4))

Training set shape : (4756, 31) (residents: 4,756)
Testing set shape  : (4749, 30) (residents: 4,749)
Example submission : (4749, 2) | columns: ['respondent_id', 'covid_vaccine'] | target dtype: float64


,respondent_id,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,census_msa,household_adults,household_children,doctor_recc_covid,opinion_covid_vacc_effective,opinion_covid_risk,opinion_covid_sick_from_vacc,employment_sector,number_chronic_med_conditions,covid_vaccine
0,1,3.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,"MSA, Principle City",3.0,2.0,0,4,4,2.0,construction,1,0
1,2,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Non-MSA,0.0,0.0,0,5,2,1.0,education,0,1
2,3,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,"MSA, Not Principle City",0.0,0.0,0,2,2,5.0,wholesale,3,0
3,4,2.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,"MSA, Not Principle City",1.0,0.0,1,3,3,2.0,NaN,1,1
